# Colab Pipeline Notebook

这个 notebook 专门为 Google Colab 准备，覆盖：

- 项目克隆
- Google Drive 挂载
- 数据集上传 / 下载 / 解压
- 依赖安装
- Hugging Face 登录与权重下载
- detection 条件构建
- SDXL inpainting LoRA 训练
- 术前多视角推理
- 结果打包下载

当前仓库默认流程：

1. `detection.prepare_dataset` 生成方形头部 `face_mask`、连续鼻嘴 `inpaint_mask`、`eye_privacy_mask`、`sanitized` 图和 manifest
2. `generation.train` 用 `sanitized + inpaint_mask` 做 SDXL inpainting LoRA 训练
3. `generation.infer` 对术前六视角图做鼻嘴局部重绘


In [ ]:
#@title 1. 检查 Colab 运行环境
!nvidia-smi || true

import os
import platform
import sys

print('python =', sys.version)
print('platform =', platform.platform())


Sun Apr  5 14:45:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#@title 2. 挂载 Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [5]:
#@title 3. 克隆项目仓库
REPO_URL = 'https://github.com/longhaoleo/myproject.git'  #@param {type:"string"}
REPO_DIR = '/content/myproject'  #@param {type:"string"}
BRANCH = ''  #@param {type:"string"}

if not os.path.exists(REPO_DIR):
    !git clone "$REPO_URL" "$REPO_DIR"
else:
    print('repo already exists:', REPO_DIR)

%cd {REPO_DIR}
if BRANCH.strip():
    !git fetch --all
    !git checkout "$BRANCH"
    !git pull --ff-only

!git rev-parse --short HEAD


Cloning into '/content/myproject'...
remote: Enumerating objects: 93, done.
remote: Counting objects: 100% (93/93), done.
remote: Compressing objects: 100% (76/76), done.
remote: Total 93 (delta 32), reused 75 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (93/93), 105.95 KiB | 2.41 MiB/s, done.
Resolving deltas: 100% (32/32), done.
/content/myproject
852d9b0


In [6]:
#@title 4. 安装依赖
%cd {REPO_DIR}

# Colab 自带的 torch 通常更匹配 GPU；这里先升级 pip，再安装项目依赖。
!python -m pip install -U pip setuptools wheel
!python -m pip install -r requirements.txt

import torch
print('torch =', torch.__version__)
print('cuda available =', torch.cuda.is_available())


/content/myproject
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 50.0 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 66.6 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
  Cloning https://github.com/ChaoningZhang/MobileSAM.git to /tmp/pip-req-build-vsw0mm9x
  Running command git clone --filter=blob:none --quiet https://github.com/ChaoningZhang/MobileSAM.git /tmp/pip-req-build-vsw0mm9x
  Resolved https://github.com/ChaoningZhang/MobileSAM.g

## 数据集准备

建议数据目录结构：

```text
<input_root>/
└── <case_id>/
    ├── 术前/
    │   ├── 1.JPG
    │   ├── 2.JPG
    │   └── ...
    └── 术后/
        ├── 1.JPG
        ├── 2.JPG
        └── ...
```

下面给两种常用方式：

- 方式 A：从 Google Drive 直接读取
- 方式 B：上传 zip 到 Colab 后解压


In [ ]:
#@title 5A. 直接使用 Google Drive 上的数据
DATA_ROOT = '/content/drive/MyDrive/deformity'  #@param {type:"string"}
print('DATA_ROOT =', DATA_ROOT)
!ls "$DATA_ROOT" | head


In [ ]:
#@title 5B. 上传 zip 到 Colab 并解压（可选）
# 如果你已经用 5A，这个 cell 可以跳过。
from google.colab import files
import os
import zipfile

UPLOAD_TO = '/content/uploads'  #@param {type:"string"}
EXTRACT_TO = '/content/datasets/deformity'  #@param {type:"string"}
os.makedirs(UPLOAD_TO, exist_ok=True)
os.makedirs(EXTRACT_TO, exist_ok=True)

uploaded = files.upload()
for name, content in uploaded.items():
    zip_path = os.path.join(UPLOAD_TO, name)
    with open(zip_path, 'wb') as f:
        f.write(content)
    if zip_path.lower().endswith('.zip'):
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(EXTRACT_TO)
        print('extracted to', EXTRACT_TO)
    else:
        print('uploaded non-zip file:', zip_path)

DATA_ROOT = EXTRACT_TO
print('DATA_ROOT =', DATA_ROOT)


In [ ]:
#@title 6. 设置输出路径和模型目录
PROJECT_ROOT = REPO_DIR
MODEL_DIR = '/content/drive/MyDrive/myproject_models'  #@param {type:"string"}
GEN_ROOT = '/content/drive/MyDrive/deformity_generation'  #@param {type:"string"}

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(GEN_ROOT, exist_ok=True)

os.environ['FD_INPUT_ROOT'] = DATA_ROOT
os.environ['FD_MODEL_DIR'] = MODEL_DIR
os.environ['FD_OUTPUT_ROOT_SAM'] = os.path.join(GEN_ROOT, 'sam')

os.environ['GEN_INPUT_ROOT'] = DATA_ROOT
os.environ['GEN_MODEL_DIR'] = MODEL_DIR
os.environ['GEN_OUTPUT_ROOT'] = GEN_ROOT

for key in ['FD_INPUT_ROOT', 'FD_MODEL_DIR', 'FD_OUTPUT_ROOT_SAM', 'GEN_INPUT_ROOT', 'GEN_MODEL_DIR', 'GEN_OUTPUT_ROOT']:
    print(key, '=', os.environ[key])


In [ ]:
#@title 7. Hugging Face 登录和权重下载
# 这里默认准备 inpainting 训练 / 推理底模。
%cd {REPO_DIR}
!python -m pip install -U huggingface-hub

HF_MODEL_ROOT = os.path.join(MODEL_DIR, 'generation')
os.makedirs(HF_MODEL_ROOT, exist_ok=True)

print('先运行下面一行完成登录，然后再执行下载命令。')
print('!huggingface-cli login')

# 取消注释后执行
# !huggingface-cli download diffusers/stable-diffusion-xl-1.0-inpainting-0.1 --local-dir "$HF_MODEL_ROOT/sdxl-inpaint"
# !huggingface-cli download diffusers/controlnet-depth-sdxl-1.0 --local-dir "$HF_MODEL_ROOT/controlnet-depth-sdxl"


In [ ]:
#@title 8. 运行 detection / SAM，生成部位 mask
# 先确保 detection/sam_mask.py 所需权重已准备好。
%cd {REPO_DIR}

# 按需三选一：
RUN_DETECT_COMPARE = False  #@param {type:"boolean"}
RUN_EYE_MASK = False  #@param {type:"boolean"}
RUN_SAM_MASK = True  #@param {type:"boolean"}

if RUN_DETECT_COMPARE:
    from detection.batch import main as run_detect_compare
    run_detect_compare()

if RUN_EYE_MASK:
    from detection.masking import main as run_eye_mask
    run_eye_mask()

if RUN_SAM_MASK:
    from detection.sam_mask import main as run_sam_mask
    run_sam_mask()


In [ ]:
#@title 9. 生成 detection 条件产物
%cd {REPO_DIR}

from detection.prepare_dataset import prepare_dataset
from generation.settings import DatasetBuildConfig, default_generation_paths

summary = prepare_dataset(
    paths=default_generation_paths(),
    config=DatasetBuildConfig(
        enable_depth_condition=False,
        apply_eye_privacy_mask=True,
    ),
)
summary


In [ ]:
#@title 10. 训练 SDXL inpainting LoRA
%cd {REPO_DIR}

from generation.train import train_lora
from generation.settings import default_generation_paths, default_lora_train_config, update_dataclass

train_config = update_dataclass(
    default_lora_train_config(),
    base_model_id=os.path.join(MODEL_DIR, 'generation', 'sdxl-inpaint'),
    resolution=1024,
    train_batch_size=1,
    gradient_accumulation_steps=4,
    max_train_steps=200,
    mixed_precision='fp16',
)

train_summary = train_lora(
    paths=default_generation_paths(),
    config=train_config,
)
train_summary


In [ ]:
#@title 11. 对术前多视角图运行推理
%cd {REPO_DIR}

from generation.infer import run_inference
from generation.settings import default_generation_paths, default_inference_config, update_dataclass

infer_config = update_dataclass(
    default_inference_config(),
    base_model_id=os.path.join(MODEL_DIR, 'generation', 'sdxl-inpaint'),
    lora_weights_path=os.path.join(GEN_ROOT, 'lora', 'final'),
    enable_depth_condition=False,
    quantization_backend='none',
    max_samples=0,
)

infer_summary = run_inference(
    paths=default_generation_paths(),
    config=infer_config,
)
infer_summary


In [ ]:
#@title 12. 打包结果并下载
import shutil
from google.colab import files

ARCHIVE_BASE = '/content/deformity_generation_results'
archive_path = shutil.make_archive(ARCHIVE_BASE, 'zip', GEN_ROOT)
print('archive =', archive_path)
files.download(archive_path)
